# DATA 298B Grounded SFT Training Orchestrator

This notebook regenerates private, grounded SFT records and then calls the existing `src.training.qlora_sft` trainer. It does not implement QLoRA in the notebook.

In [ ]:
# ---- Edit this cell only ----
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-backend'
MODEL = 'ministral'          # ministral | gemma | granite | qwen
MODE = 'pilot'               # pilot | full
MOUNT_DRIVE = True
HF_LOGIN_REQUIRED = True
RUN_DRY_RUN_VALIDATION = False
PUSH_PRIVATE_ADAPTER = False
HF_ADAPTER_REPO = ''
PILOT_CASES = 100
PILOT_STEPS = 75
FULL_STEPS = 1000
SEED = 42

# Set these two Drive paths. They are used only to generate the private dataset.
PROCESSED_PARQUET_PATH = ''
PATTERNS_PATH = ''

assert MODEL in {'ministral', 'gemma', 'granite', 'qwen'}
assert MODE in {'pilot', 'full'}
if PUSH_PRIVATE_ADAPTER: assert HF_ADAPTER_REPO, 'Set HF_ADAPTER_REPO first'
print({'MODEL': MODEL, 'MODE': MODE, 'BRANCH': REPO_BRANCH})

In [ ]:
# Clone or pull the feature branch.
from pathlib import Path
import os, subprocess

REPO_DIR = Path('/content/Graph-Based-Fraud-Intelligence-Platform')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR)

In [ ]:
# Mount Drive and choose a new adapter directory.
from pathlib import Path
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/DATA298B')
else:
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'colab_local'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DRIVE_ROOT / 'adapters' / f'{MODEL}_{MODE}_grounded'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
print('adapter output:', ADAPTER_DIR)

In [ ]:
# Install requirements and GPU dependencies.
%pip install -q -r requirements.txt
%pip install -q accelerate bitsandbytes sentencepiece huggingface_hub
print('dependencies installed')

In [ ]:
# Secure Hugging Face login. No token is stored in this notebook.
if HF_LOGIN_REQUIRED:
    from huggingface_hub import login
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    login(token=token or None, add_to_git_credential=False)

In [ ]:
# Fail early if the GPU is clearly undersized.
import importlib.metadata as md
import importlib.util
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. Enable a Colab GPU runtime.')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
required_gb = 20 if MODEL == 'ministral' else 28
print('GPU:', torch.cuda.get_device_name(0))
print(f'VRAM: {vram_gb:.1f} GiB; threshold: {required_gb} GiB')
print('CUDA:', torch.version.cuda)
for name in ['torch', 'transformers', 'bitsandbytes']:
    print(name, md.version(name) if importlib.util.find_spec(name) else 'not installed')
if vram_gb < required_gb:
    raise RuntimeError(f'Insufficient VRAM: {vram_gb:.1f} GiB; approximately {required_gb} GiB required.')

In [ ]:
# Verify the registry and generate grounded private SFT data.
import json, sys, yaml
sys.path.insert(0, str(REPO_DIR))
registry = yaml.safe_load((REPO_DIR / 'configs/models/registry.yaml').read_text())
if MODEL not in registry:
    raise KeyError(f'{MODEL!r} is not in the model registry')
SPEC = registry[MODEL]
print('selected:', SPEC)

if not PROCESSED_PARQUET_PATH or not PATTERNS_PATH:
    raise ValueError('Set PROCESSED_PARQUET_PATH and PATTERNS_PATH in the first cell.')
PARQUET_PATH = Path(PROCESSED_PARQUET_PATH)
PATTERNS_FILE = Path(PATTERNS_PATH)
if not PARQUET_PATH.exists(): raise FileNotFoundError(PARQUET_PATH)
if not PATTERNS_FILE.exists(): raise FileNotFoundError(PATTERNS_FILE)

from src.aml.dataset import build_cases_from_processed_data, to_sft_record
cases = build_cases_from_processed_data(PARQUET_PATH, PATTERNS_FILE, Path('/tmp/aml_cases_private.jsonl'), limit=100)
DATASET = DRIVE_ROOT / 'datasets' / 'aml_sft_grounded.jsonl'
DATASET.parent.mkdir(parents=True, exist_ok=True)
with DATASET.open('w') as f:
    for case in cases: f.write(json.dumps(to_sft_record(case)) + '\n')
print('grounded cases:', len(cases))
print('private SFT dataset:', DATASET)
if len(cases) != 100: raise RuntimeError(f'Expected 100 cases, got {len(cases)}')

In [ ]:
# Verify targets are populated and inference context remains label-free.
records = [json.loads(line) for line in DATASET.open() if line.strip()]
reports = [json.loads(r['messages'][2]['content']) for r in records]
contexts = [r['messages'][1]['content'].lower() for r in records]
for forbidden in ['is_laundering', 'fraud_label', 'pattern_type', 'ground_truth']:
    assert all(forbidden not in context for context in contexts), forbidden
assert all(r['evidence'] and r['recommended_actions'] and r['summary'] for r in reports)
print('records:', len(records))
print('sample evidence:', reports[0]['evidence'])
print('sample actions:', reports[0]['recommended_actions'])
print('sample summary:', reports[0]['summary'])

In [ ]:
# Optional repository dry run.
if RUN_DRY_RUN_VALIDATION:
    subprocess.run(['python', 'scripts/run_298b_dry_run.py'], check=True)
else:
    print('dry run skipped')

TRAIN_DATASET = DATASET
STEPS = PILOT_STEPS if MODE == 'pilot' else FULL_STEPS
print('training dataset:', TRAIN_DATASET, 'steps:', STEPS)

In [ ]:
# Call the existing trainer. No QLoRA implementation is duplicated here.
from src.training.qlora_sft import train
metrics = train(MODEL, TRAIN_DATASET, ADAPTER_DIR, max_steps=STEPS, max_samples=PILOT_CASES if MODE == 'pilot' else None, seed=SEED)
print('training metrics:', metrics)
print('adapter directory:', ADAPTER_DIR)

In [ ]:
# Verify the saved adapter.
from peft import PeftConfig
required = ['adapter_config.json', 'tokenizer_config.json', 'training_run.json', 'loss_history.json']
missing = [name for name in required if not (ADAPTER_DIR / name).exists()]
if not any((ADAPTER_DIR / name).exists() for name in ['adapter_model.safetensors', 'adapter_model.bin']): missing.append('adapter model')
if missing: raise FileNotFoundError(f'Missing training artifacts: {missing}')
print('adapter loads; base model:', PeftConfig.from_pretrained(str(ADAPTER_DIR)).base_model_name_or_path)

In [ ]:
# One post-training AML inference example.
from src.aml.backend import TransformersBackend
from src.aml.agent import AMLAgent
from src.aml.schemas import InvestigationCase
case_payload = json.loads(next(TRAIN_DATASET.open()))
case = InvestigationCase(case_id=case_payload['case_id'], context=json.loads(case_payload['messages'][1]['content']))
backend = TransformersBackend(SPEC['model_id'], revision=SPEC['revision'], adapter_path=str(ADAPTER_DIR), model_kwargs={'load_in_4bit': True, 'device_map': 'auto'})
trace = AMLAgent(backend, max_tool_calls=3, artifact_dir=str(ADAPTER_DIR / 'inference_traces')).investigate(case, model_family=MODEL, variant='finetuned_grounded', seed=SEED)
print(json.dumps(trace['report'], indent=2, default=str))
print('trace directory:', ADAPTER_DIR / 'inference_traces')

In [ ]:
# Optional private Hugging Face upload.
if PUSH_PRIVATE_ADAPTER:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HF_ADAPTER_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=HF_ADAPTER_REPO, repo_type='model')
    print('uploaded:', HF_ADAPTER_REPO)
else:
    print('Hub upload skipped; adapter remains on Drive')
print('FINAL ADAPTER:', ADAPTER_DIR)
print('FINAL DATASET:', DATASET)